In [1]:
# For reading data
import os
import numpy as np
from xml.dom import minidom

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

# For visualizing
import plotly.express as px
from torchvision.io import read_image
from torchvision import tv_tensors
from torchvision.transforms.v2 import functional as F
!pip install torch opencv-python
# For model building
import torch
import torch.nn as nn
import torchvision
from torchvision.models.detection import FasterRCNN
# for videos

import cv2 as cv


In [3]:
class BaseballVideos(torch.utils.data.Dataset):
    def __init__(self, root=None, transforms=None):
        self.root = root
        self.transforms = transforms
        # load all image files, sorting them to
        # ensure that they are aligned
        if root==None:
            self.vids = list(sorted([os.path.join("Model Data2", i) for i in os.listdir(os.path.join(os.path.curdir, "Model Data2")) if '.mov' in i]))[0:5]
            self.notes = list(sorted([os.path.join("Model Data2", i) for i in os.listdir(os.path.join(os.path.curdir, "Model Data2")) if '.xml' in i]))[0:5]
            if len(self.vids)!=len(self.notes):
                raise RuntimeError("Mismatch of annotation files and video files.\nPlease confirm that you have one annotation file for each video and try again.")
        imgs = []
        notes = []
        index = 0
        for i, k in zip(self.vids, self.notes):
            cap = cv.VideoCapture(i)
            note = minidom.parse(k)
            ret = True
            frame_count = 0
            print(index)
            index += 1
            while ret:
                ret, frame = cap.read()
                if ret:
                    frame_count += 1
                    frame = np.moveaxis(frame, -1, 0) # Pivot image so color channels first, then H then W
                    frame = torch.from_numpy(frame).float() / 255.0  # Convert to float and scale to [0, 1]
                    imgs.append(frame)
                    canvas_size = list(frame.shape[1:])

                for f in range(frame_count):
                    frame_i = [j for j in note.getElementsByTagName("box") if int(j.attributes['frame'].value)==f]
                    boxes = []
                    labels = []
                    areas = []
                    movings = []

                    for j in frame_i:
                        attrs = j.getElementsByTagName('attribute')
                        #print(attrs[0])

                        if attrs is not None:
                            print(attrs[0].firstChild.data)
                            moving = attrs[0].firstChild.data == 'true'
                        
                        else:
                            moving = False  # default if attribute is missing

                        print(moving)
                        if moving:
                            xtl = float(j.attributes['xtl'].value)
                            ytl = float(j.attributes['ytl'].value)
                            xbr = float(j.attributes['xbr'].value)
                            ybr = float(j.attributes['ybr'].value)
                            box = (xtl, ytl, xbr, ybr)

                            label = 1  # baseball class index
                            area = (xbr - xtl) * (ybr - ytl)

                            boxes.append(box)
                            labels.append(label)
                            areas.append(area)
                            #movings.append(moving)

                    target = {}
                    if len(boxes) == 0:
                        boxes_tensor = torch.zeros((0, 4), dtype=torch.float32)
                        labels_tensor = torch.zeros((0,), dtype=torch.int64)
                    else:
                        boxes_tensor = torch.tensor(boxes, dtype=torch.float32)
                        labels_tensor = torch.tensor(labels, dtype=torch.int64)
                    target["boxes"] = tv_tensors.BoundingBoxes(boxes_tensor, format="XYXY", canvas_size=canvas_size)                    
                    target["labels"] = labels_tensor
                    target["area"] = areas
                    #target["moving"] = movings

                    notes.append(target)
              
        self.imgs = imgs
        self.notes = notes

            # target = {}
            # target["boxes"] = tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=F.get_size(img))
            # target["masks"] = tv_tensors.Mask(masks)
            # target["labels"] = labels
            # target["image_id"] = image_id
            # target["area"] = area
            # target["iscrowd"] = iscrowd
        self.imgs
        self.notes

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):

        img = self.imgs[idx]
        target = self.notes[idx]

        if self.transforms is not None:
            img, target = self.transforms(img, target)

        return img, target

dataset = BaseballVideos()

FileNotFoundError: [WinError 3] The system cannot find the path specified: '.\\Model Data2'

In [ ]:
class TestBaseballVideos(torch.utils.data.Dataset):
    def __init__(self, root=None, transforms=None):
        self.root = root
        self.transforms = transforms
        # load all image files, sorting them to
        # ensure that they are aligned
        if root==None:
            self.vids = list(sorted([os.path.join("Test Videos", i) for i in os.listdir(os.path.join(os.path.curdir, "Test Videos")) if '.mov' in i]))[0:5]
            self.notes = list(sorted([os.path.join("Test Videos", i) for i in os.listdir(os.path.join(os.path.curdir, "Test Videos")) if '.xml' in i]))[0:5]
            if len(self.vids)!=len(self.notes):
                raise RuntimeError("Mismatch of annotation files and video files.\nPlease confirm that you have one annotation file for each video and try again.")
        imgs = []
        notes = []
        index = 0
        for i, k in zip(self.vids, self.notes):
            cap = cv.VideoCapture(i)
            note = minidom.parse(k)
            ret = True
            frame_count = 0
            print(index)
            index += 1
            while ret:
                ret, frame = cap.read()
                if ret:
                    frame_count += 1
                    frame = np.moveaxis(frame, -1, 0) # Pivot image so color channels first, then H then W
                    frame = torch.from_numpy(frame).float() / 255.0  # Convert to float and scale to [0, 1]
                    imgs.append(frame)
                    canvas_size = list(frame.shape[1:])

                for f in range(frame_count):
                    frame_i = [j for j in note.getElementsByTagName("box") if int(j.attributes['frame'].value)==f]
                    boxes = []
                    labels = []
                    areas = []
                    movings = []

                    for j in frame_i:
                        attrs = j.getElementsByTagName('attribute')
                        #print(attrs[0])

                        if attrs is not None:
                            print(attrs[0].firstChild.data)
                            moving = attrs[0].firstChild.data == 'true'
                        
                        else:
                            moving = False  # default if attribute is missing

                        print(moving)
                        if moving:
                            xtl = float(j.attributes['xtl'].value)
                            ytl = float(j.attributes['ytl'].value)
                            xbr = float(j.attributes['xbr'].value)
                            ybr = float(j.attributes['ybr'].value)
                            box = (xtl, ytl, xbr, ybr)

                            label = 1  # baseball class index
                            area = (xbr - xtl) * (ybr - ytl)

                            boxes.append(box)
                            labels.append(label)
                            areas.append(area)
                            #movings.append(moving)

                    target = {}
                    if len(boxes) == 0:
                        boxes_tensor = torch.zeros((0, 4), dtype=torch.float32)
                        labels_tensor = torch.zeros((0,), dtype=torch.int64)
                    else:
                        boxes_tensor = torch.tensor(boxes, dtype=torch.float32)
                        labels_tensor = torch.tensor(labels, dtype=torch.int64)
                    target["boxes"] = tv_tensors.BoundingBoxes(boxes_tensor, format="XYXY", canvas_size=canvas_size)
                    target["labels"] = labels_tensor
                    target["area"] = areas
                    #target["moving"] = movings

                    notes.append(target)
              
        self.imgs = imgs
        self.notes = notes

            # target = {}
            # target["boxes"] = tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=F.get_size(img))
            # target["masks"] = tv_tensors.Mask(masks)
            # target["labels"] = labels
            # target["image_id"] = image_id
            # target["area"] = area
            # target["iscrowd"] = iscrowd
        self.imgs
        self.notes

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):

        img = self.imgs[idx]
        target = self.notes[idx]

        if self.transforms is not None:
            img, target = self.transforms(img, target)

        return img, target

test_dataset = TestBaseballVideos()

In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))

In [ ]:
train_dl = torch.utils.data.DataLoader(dataset,
                                       batch_size = 1,
                                       shuffle = True,
                                       collate_fn = collate_fn,
                                       pin_memory=False,
                                       num_workers=0)

test_dl = torch.utils.data.DataLoader(test_dataset,
                                       batch_size = 1,
                                       shuffle = True,
                                       collate_fn = collate_fn,
                                       pin_memory=False,
                                       num_workers=0)

In [ ]:
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True, weights="DEFAULT")

num_classes = 2

in_features = model.roi_heads.box_predictor.cls_score.in_features

model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

In [ ]:

if torch.xpu.is_available():
    device = torch.device("xpu")
    print(f"Using device: {torch.xpu.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("XPU not found, falling back to CPU.")

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr = .001, momentum=.9, weight_decay=.0005)

num_epochs = 1

In [ ]:
from tqdm.notebook import tqdm

model.to(device)

for epoch in range(num_epochs):
    # --- TRAINING PHASE ---
    model.train()
    train_iter = tqdm(train_dl, desc=f"Epoch {epoch+1}/{num_epochs}")
    for images, targets in train_iter:
        # Move data to Intel iGPU
        images = list(image.to(device) for image in images)
        targets = [
            {k: v.to(device) if hasattr(v, 'to') else v for k, v in t.items()}
            for t in targets
        ]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        train_iter.set_postfix({'loss': losses.item()})

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()


In [ ]:
# --- TESTING/EVALUATION PHASE ---
model.eval()
with torch.no_grad():
    for images, targets in test_dl:
        images = list(image.to(device) for image in images)
        # Targets aren't sent to the model in eval mode, 
        # but you keep them to compare against predictions later.
        predictions = model(images) 
    

In [ ]:
data = iter(test_dl).__next__()

In [ ]:
img = data[0][0]
boxes = data[0][1]["boxes"]
labels = data[0][1]["labels"]

In [ ]:
output = model([img.to(device)])

In [ ]:
output

In [ ]:
out_bbox = output[0]["boxes"]
out_scores = output[0]["scores"]

In [ ]:
keep = torchvision.ops.nms(out_bbox, out_scores, 0.45)

In [ ]:
out_bbox.shape, keep.shape

In [ ]:
im = (img.permute(1,2,0).cpu().detach().numpy() * 255).astype('uint8')

In [ ]:
im

In [ ]:
torch.save(model.state_dict(), "baseball_model2.pt")
print("model saved")